# Weather — Ingestion & Cleaning

## Objective
Merge the temperature and solar-irradiance reanalysis sources for a single Northern
Italy grid point into one hourly series, clean it, and derive the clearness index
the solar-irradiation and capture-rate models are fit on.

- **Site**: `45.5N, 11.25E` — a Po Valley grid point in the NORD bidding zone,
  roughly 115 km from Bologna proper. The `bologna` name in file paths is kept
  for compatibility with the rest of the pipeline, but the coordinates above are
  the ones actually used.
- **ERA5** (`t2m`, `ssrd`): temperature and downward solar radiation, 1995-2025, UTC.
- **CAMS** (`GHI`, `CLEAR_SKY_GHI`): observed and theoretical clear-sky global
  horizontal irradiance, 2005-2025, UTC, timestamped at the end of each hour.

## Pipeline Position

| Inputs | This notebook | Outputs |
|---|---|---|
| `Data/Raw/reanalysis-era5-single-levels-timeseries-sfc7rupio5o.nc` | Merge -> clean -> engineer | `Data/Cleaned/df_bologna_cleaned.parquet` |
| `Data/Raw/cams_solar_irradiance_ts.nc` | | `Data/Interim/df_bologna_full.parquet` (merged checkpoint) |

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr


def find_project_root() -> Path:
    """Walk up from the cwd until the directory holding Code/ and Data/ is found."""
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "Code").is_dir() and (cand / "Data").is_dir():
            return cand
    raise RuntimeError(f"Project root not found above {here}")


ROOT = find_project_root()
ERA5_NC       = ROOT / "Data" / "Raw"     / "reanalysis-era5-single-levels-timeseries-sfc7rupio5o.nc"
CAMS_NC       = ROOT / "Data" / "Raw"     / "cams_solar_irradiance_ts.nc"
INTERIM_PARQ  = ROOT / "Data" / "Interim" / "df_bologna_full.parquet"
OUTPUT_CLEAN  = ROOT / "Data" / "Cleaned" / "df_bologna_cleaned.parquet"

## Load ERA5 (Temperature & Radiation) and CAMS (Irradiance)

In [2]:
era5 = xr.open_dataset(ERA5_NC).to_dataframe().reset_index()
era5 = era5.rename(columns={"valid_time": "time"})
era5["SSRD_Wm2"] = era5["ssrd"] / 3600   # J/m^2 accumulated over 1h -> W/m^2

print(f"ERA5 rows  : {len(era5):,}")
print(f"ERA5 range : {era5['time'].min()}  ->  {era5['time'].max()}")

ERA5 rows  : 271,752
ERA5 range : 1995-01-01 00:00:00  ->  2025-12-31 23:00:00


In [3]:
cams_ds = xr.open_dataset(CAMS_NC)
cams = cams_ds.to_dataframe().reset_index()[["time", "CLEAR_SKY_GHI", "GHI"]]

print(f"CAMS rows  : {len(cams):,}")
print(f"CAMS range : {cams['time'].min()}  ->  {cams['time'].max()}")

CAMS rows  : 184,080


CAMS range : 2005-01-01 01:00:00  ->  2026-01-01 00:00:00


## Merge

### Decision Log

| # | Decision | Alternatives | Rationale | Impact |
|---|---|---|---|---|
| D1 | Inner join on `time` | Outer join, keep full ERA5 range | CAMS irradiance only starts in 2005; an outer join would leave 1995-2004 with no GHI, unusable for the solar/capture-rate models that need both series together | Restricts the sample to 2005-2025 (still 21 years); 1995-2004 temperature-only data is dropped, a deliberate common-window choice, not an oversight |
| D2 | Join on raw UTC timestamp, no offset | Shift one series by an hour | Both sources are UTC; CAMS documents its timestamp as end-of-hour, and ERA5's accumulated `ssrd` in this product follows the same end-of-hour convention, so the timestamps already line up | No adjustment needed; recorded here so the alignment is explicit rather than assumed |

In [4]:
# D1: inner join. D2: joined on the raw UTC timestamp, no offset --
# both sources are UTC and already share the same end-of-hour convention.
df = pd.merge(era5[["time", "latitude", "longitude", "t2m", "SSRD_Wm2"]], cams, on="time", how="inner")

print(f"Merged rows : {len(df):,}")
print(f"Merged range: {df['time'].min()}  ->  {df['time'].max()}")
print(f"Dropped from ERA5 (pre-2005, no CAMS overlap): {len(era5) - len(df):,} rows")

Merged rows : 184,079
Merged range: 2005-01-01 01:00:00  ->  2025-12-31 23:00:00
Dropped from ERA5 (pre-2005, no CAMS overlap): 87,673 rows


## Save the Merged Checkpoint

In [5]:
INTERIM_PARQ.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(INTERIM_PARQ, index=False)
print(f"[SAVED] {INTERIM_PARQ}  ({len(df):,} rows)")

[SAVED] C:\Users\LucasMonero\OneDrive - EloGroup\Documentos\data projects\Master Thesis\Project\Data\Interim\df_bologna_full.parquet  (184,079 rows)


## Cleaning & Feature Engineering

Unit conversion and gap-filling come first; the clearness index is derived
**last**, from the filled `GHI`, so it never carries stale missing values.

In [6]:
df["t2m"] = df["t2m"] - 273.15   # Kelvin -> Celsius
df = df.sort_values("time", ignore_index=True)

n_missing_ghi = df["GHI"].isna().sum()
print(f"Missing GHI before fill: {n_missing_ghi}  ({n_missing_ghi / len(df):.3%} of rows)")

Missing GHI before fill: 53  (0.029% of rows)


### Decision Log — Imputation Rules

| # | Decision | Rationale | Impact |
|---|---|---|---|
| D3 | Night hours (`CLEAR_SKY_GHI == 0`) -> `GHI = 0` | Physically certain: no sunlight, no irradiance | Resolves the bulk of missing hours with no assumption |
| D4 | Partial-day gaps -> `CLEAR_SKY_GHI * daily mean clearness ratio` | Preserves that day's actual cloud conditions from its own observed hours | Assumes the day's clearness is uniform across the missing hours |
| D5 | Whole-day gaps -> `CLEAR_SKY_GHI` directly (assumed clear sky) | No same-day observation exists to estimate cloudiness from | The strongest assumption here: biases the clearness-index distribution slightly toward "clear" on these days — a known limitation for the solar model fit downstream |

In [7]:
df["date"] = df["time"].dt.date

# D3: night is certain
df.loc[df["GHI"].isna() & (df["CLEAR_SKY_GHI"] == 0), "GHI"] = 0

# D4/D5: fill remaining gaps from the day's own clearness ratio, or clear-sky if the whole day is missing
ratio = (df["GHI"] / df["CLEAR_SKY_GHI"]).replace([np.inf, -np.inf], np.nan)
daily_mean_ratio = ratio.groupby(df["date"]).transform("mean")
df["GHI"] = df["GHI"].fillna(df["CLEAR_SKY_GHI"] * daily_mean_ratio)   # D4
df["GHI"] = df["GHI"].fillna(df["CLEAR_SKY_GHI"])                      # D5

df = df.drop(columns=["date"])
print(f"Missing GHI after fill: {df['GHI'].isna().sum()}")

Missing GHI after fill: 0


In [8]:
# Clearness index, derived last so it reflects the filled GHI.
# Night hours (CLEAR_SKY_GHI == 0) give a structurally undefined 0/0 ratio,
# not a missing value -- kept as NaN and documented, rather than filled.
df["GHI_index"] = (df["GHI"] / df["CLEAR_SKY_GHI"]).replace([np.inf, -np.inf], np.nan)

n_night = (df["CLEAR_SKY_GHI"] == 0).sum()
print(f"GHI_index NaN        : {df['GHI_index'].isna().sum():,}")
print(f"Night hours (0/0)    : {n_night:,}")
assert df["GHI_index"].isna().sum() == n_night, "GHI_index has NaN beyond the structural night-time 0/0 case"
print(f"GHI_index range (daytime): [{df['GHI_index'].min():.4f}, {df['GHI_index'].max():.4f}]")

GHI_index NaN        : 84,052
Night hours (0/0)    : 84,052


GHI_index range (daytime): [0.0627, 1.0031]


## Save & Handoff

### Conclusion
- Hourly temperature and irradiance series, 2005-2025, UTC, at the modelled grid point.
- `GHI_index` is derived after imputation, not before, so it carries no stale
  missing values. Its only remaining NaNs are the structurally undefined
  night-time 0/0 ratio (`CLEAR_SKY_GHI == 0`), which is expected and not a data gap.
- Constant `latitude`/`longitude` columns dropped (documented in Objective above, not carried as data).

**Next notebook can rely on:** `Data/Cleaned/df_bologna_cleaned.parquet`, keyed on
`time` (UTC), for the solar irradiation, temperature, and capture-rate models.

In [9]:
assert df["time"].is_monotonic_increasing and not df["time"].duplicated().any()
assert df["GHI_index"].isna().sum() == (df["CLEAR_SKY_GHI"] == 0).sum()

df = df.drop(columns=["latitude", "longitude"])

OUTPUT_CLEAN.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_CLEAN, index=False)
print(f"[SAVED] {OUTPUT_CLEAN}")
print(f"  Final shape: {df.shape}")

[SAVED] C:\Users\LucasMonero\OneDrive - EloGroup\Documentos\data projects\Master Thesis\Project\Data\Cleaned\df_bologna_cleaned.parquet
  Final shape: (184079, 6)
